# Data Cleaning and Preparation

Before starting any analysis, it's critical to understand the dataset deeply. Poor data understanding leads to wrong conclusions. In this section, we will inspect the dataset, check data types, handle missing values, and extract new features.

### 1. Load the Dataset and Inspect Basic Information

In [8]:
import pandas as pd
from IPython.display import display
# A sample of the dataset is used to improve performance
# and avoid memory issues while maintaining meaningful analysis

df = pd.read_csv(
    '../data/raw/flights_sample_3m.csv',
    nrows=300000
)

print('First 5 rows:')
display(df.head())

print('\nDataset Info:')
df.info()

print('\nDataset Shape:', df.shape)

First 5 rows:


,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,0.0,186.0,176.0,153.0,1065.0,NaN,NaN,NaN,NaN,NaN
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,0.0,235.0,236.0,189.0,1399.0,NaN,NaN,NaN,NaN,NaN
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,0.0,118.0,112.0,87.0,680.0,NaN,NaN,NaN,NaN,NaN
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,0.0,181.0,182.0,153.0,985.0,NaN,NaN,NaN,NaN,NaN



Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 300000 entries, 0 to 299999
Data columns (total 32 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   FL_DATE                  300000 non-null  object 
 1   AIRLINE                  300000 non-null  object 
 2   AIRLINE_DOT              300000 non-null  object 
 3   AIRLINE_CODE             300000 non-null  object 
 4   DOT_CODE                 300000 non-null  int64  
 5   FL_NUMBER                300000 non-null  int64  
 6   ORIGIN                   300000 non-null  object 
 7   ORIGIN_CITY              300000 non-null  object 
 8   DEST                     300000 non-null  object 
 9   DEST_CITY                300000 non-null  object 
 10  CRS_DEP_TIME             300000 non-null  int64  
 11  DEP_TIME                 292157 non-null  float64
 12  DEP_DELAY                292153 non-null  float64
 13  TAXI_OUT                 292045 non-null  fl

### 2. Identifying and Handling Missing Values

First, we check where the missing values are located.

In [9]:
print('Missing Values:\n', df.isnull().sum())

Missing Values:
 FL_DATE                         0
AIRLINE                         0
AIRLINE_DOT                     0
AIRLINE_CODE                    0
DOT_CODE                        0
FL_NUMBER                       0
ORIGIN                          0
ORIGIN_CITY                     0
DEST                            0
DEST_CITY                       0
CRS_DEP_TIME                    0
DEP_TIME                     7843
DEP_DELAY                    7847
TAXI_OUT                     7955
WHEELS_OFF                   7955
WHEELS_ON                    8061
TAXI_IN                      8061
CRS_ARR_TIME                    0
ARR_TIME                     8061
ARR_DELAY                    8643
CANCELLED                       0
CANCELLATION_CODE          292014
DIVERTED                        0
CRS_ELAPSED_TIME                2
ELAPSED_TIME                 8643
AIR_TIME                     8643
DISTANCE                        0
DELAY_DUE_CARRIER          246614
DELAY_DUE_WEATHER          2466

**Assumptions for Data Cleaning:**

1. **Missing Delay Causes = 0 Minutes:** For specific delay columns (`DELAY_DUE_CARRIER`, `DELAY_DUE_WEATHER`, etc.), missing values mean no delay occurred for that reason. We will replace these NaNs with `0`.
2. **Missing Arrival Delay = Unusable Record:** Since our main goal is analyzing delays, rows missing the actual `ARR_DELAY` value provide no analytical benefit and cannot be reliably imputed. We will drop these rows.

In [10]:
# Delay-related columns
delay_cols = [
    'DELAY_DUE_CARRIER',
    'DELAY_DUE_WEATHER',
    'DELAY_DUE_NAS',
    'DELAY_DUE_SECURITY',
    'DELAY_DUE_LATE_AIRCRAFT'
]

# Replace missing delay causes with 0
for col in delay_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

# Remove rows where arrival delay is missing
if 'ARR_DELAY' in df.columns:
    df = df[df['ARR_DELAY'].notna()]

### 3. Check for Duplicates

**Assumption:** Duplicate rows are data entry errors and can artificially inflate frequency. We will remove them.

In [11]:
print('Duplicate Rows:', df.duplicated().sum())

# Remove duplicates
df = df.drop_duplicates()

Duplicate Rows: 0


### 4. Convert Data Types and Feature Engineering

**Assumption:** Flight delays are highly influenced by temporal patterns (time of year, day of week). We will convert `FL_DATE` to datetime and extract useful features like `MONTH` and `DAY_NAME`.

In [12]:
# Convert flight date column to datetime
if 'FL_DATE' in df.columns:
    df['FL_DATE'] = pd.to_datetime(df['FL_DATE'])

# Create new features
if 'FL_DATE' in df.columns:
    df['MONTH'] = df['FL_DATE'].dt.month
    df['DAY_NAME'] = df['FL_DATE'].dt.day_name()

### 5. Final Checks and Saving the Dataset

In [13]:
print('\nCleaned Dataset Info:')
df.info()

print('\nFinal Shape:', df.shape)

# Save the cleaned dataset to CSV in chunks to avoid memory issues
df.to_csv('../data/cleaned/cleaned_flights_dataset.csv', index=False)

print('\nData cleaning completed successfully!')


Cleaned Dataset Info:
<class 'pandas.core.frame.DataFrame'>
Index: 291357 entries, 0 to 299999
Data columns (total 34 columns):
 #   Column                   Non-Null Count   Dtype         
---  ------                   --------------   -----         
 0   FL_DATE                  291357 non-null  datetime64[ns]
 1   AIRLINE                  291357 non-null  object        
 2   AIRLINE_DOT              291357 non-null  object        
 3   AIRLINE_CODE             291357 non-null  object        
 4   DOT_CODE                 291357 non-null  int64         
 5   FL_NUMBER                291357 non-null  int64         
 6   ORIGIN                   291357 non-null  object        
 7   ORIGIN_CITY              291357 non-null  object        
 8   DEST                     291357 non-null  object        
 9   DEST_CITY                291357 non-null  object        
 10  CRS_DEP_TIME             291357 non-null  int64         
 11  DEP_TIME                 291357 non-null  float64       
 12